# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ariba86/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

print("Setup done!")

Setup done!


In [2]:
path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

con.sql(f"""
    CREATE OR REPLACE VIEW march_data AS
    SELECT * FROM read_parquet('{path}')
""")

print("View ready!")

View ready!


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row** = one content page, for one client, on one report date (day-level grain).

**Table used:** fact_content_daily_performance, filtered to month=2026-03 (a mid-panel month).

**Time window:** March 1, 2026 to March 31, 2026 (2026-03).

**What I'll predict/rank:** whether a page's next-day GSC click-through rate (clicks / impressions) will be high or low — a proxy for search performance, used for decision support only.

**Excluded on purpose:** GA4 (Google Analytics) columns — this lane focuses only on Google Search Console (GSC) signals, so ga4_* fields are out of scope here.
**Verified:** In month=2026-03, the table has 9,841,378 rows, all unique on
(client_hash_id, content_hash_id, report_date) — confirming the stated grain.
The data spans 2026-03-01 to 2026-03-31. Of these rows, only 3,611,061 (~36.7%)
have gsc_data_available IS TRUE, meaning most rows in a day have no real GSC
signal recorded for that content item.

In [3]:
con.sql("SELECT * FROM march_data LIMIT 5").show()


┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Context (identifiers, not model inputs):**
- client_hash_id — identifies which client the row belongs to
- content_hash_id — identifies which page the row belongs to
- report_date — the day of the observation

**Features (known at decision time, used as model inputs):**
- gsc_impressions — how many times the page appeared in Google Search that day
- gsc_clicks — how many times it was clicked that day
- gsc_avg_position — average ranking position in search results that day
- gsc_data_available — whether GSC data was actually recorded for this row

**Label (what we want to predict — the target):**
- next-day CTR (click-through rate) = next day's gsc_clicks / gsc_impressions,
  computed from the FOLLOWING day's row for the same client + content.
  This is a proxy for "search performance," not a guarantee.

**Excluded on purpose:**
- All ga4_* columns (ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions,
  ga4_total_engagement_sec) — out of scope for this GSC-focused lane.
- All sessions_* and ai_* columns — these belong to the Analytics/AI-traffic lane,
  not this one.
- gsc_sum_position — redundant with gsc_avg_position (sum is just avg × impressions),
  so it adds no new information.

In [7]:
con.sql("""
    SELECT
        client_hash_id, content_hash_id, report_date,
        gsc_impressions, gsc_clicks, gsc_avg_position, gsc_data_available
    FROM march_data
    LIMIT 3
""").show()

┌─────────────────────────┬──────────────────────────┬─────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ gsc_impressions │ gsc_clicks │ gsc_avg_position │ gsc_data_available │
│         varchar         │         varchar          │    date     │      int64      │   int64    │      double      │      boolean       │
├─────────────────────────┼──────────────────────────┼─────────────┼─────────────────┼────────────┼──────────────────┼────────────────────┤
│ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │ 2026-03-01  │              20 │          0 │             3.35 │ true               │
│ client_73cda7b4e4f265ea │ content_05597932fe4da067 │ 2026-03-01  │               1 │          0 │              0.0 │ true               │
│ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │ 2026-03-01  │             125 │          1 │            4.928 │ true               │
└───────────────────

In [8]:
features_df = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        -- Feature 1: yesterday's impressions
        LAG(gsc_impressions, 1) OVER (
            PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
        ) AS impressions_lag1,

        -- Feature 2: yesterday's clicks
        LAG(gsc_clicks, 1) OVER (
            PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
        ) AS clicks_lag1,

        -- Feature 3: yesterday's average position
        LAG(gsc_avg_position, 1) OVER (
            PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
        ) AS position_lag1,

        -- Feature 4: day of week (0=Monday ... 6=Sunday)
        DAYOFWEEK(report_date) AS day_of_week,

        -- Feature 5: was data available yesterday
        LAG(gsc_data_available, 1) OVER (
            PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
        ) AS gsc_available_lag1,

        -- LABEL: today's CTR (what we want to predict)
        CASE
            WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions
            ELSE NULL
        END AS today_ctr

    FROM march_data
    WHERE gsc_data_available IS TRUE
""").df()

print(features_df.shape)
features_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(3611061, 9)


,client_hash_id,content_hash_id,report_date,impressions_lag1,clicks_lag1,position_lag1,day_of_week,gsc_available_lag1,today_ctr
0,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,2026-03-01,<NA>,<NA>,NaN,0,<NA>,0.000000
1,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,2026-03-02,28,0,11.178571,1,True,0.000000
2,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,2026-03-03,19,0,11.368421,2,True,0.000000
3,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,2026-03-04,35,0,14.314286,3,True,0.030303
4,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,2026-03-05,33,1,12.212121,4,True,0.027027
5,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,2026-03-06,37,1,11.918919,5,True,0.000000
6,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,2026-03-07,23,0,10.652174,6,True,0.000000
7,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,2026-03-08,39,0,13.333333,0,True,0.000000
8,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,2026-03-09,17,0,11.764706,1,True,0.000000
9,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,2026-03-10,19,0,11.473684,2,True,0.000000


**Five features (all knowable at the decision moment):**

1. **impressions_lag1** (yesterday's GSC impressions) — knowable at the decision
   moment because it's yesterday's already-recorded value; today's number isn't
   needed to compute it.

2. **clicks_lag1** (yesterday's GSC clicks) — knowable because it comes from a
   completed past day, not from today or the future.

3. **position_lag1** (yesterday's average search position) — knowable because
   it reflects where the page ranked on a day that has already closed out.

4. **day_of_week** (Monday–Sunday) — knowable trivially; the calendar date is
   known in advance, before any outcome happens.

5. **gsc_available_lag1** (was GSC data recorded yesterday) — knowable because
   it's a flag from a past, already-observed row.

**Label: today_ctr** = today's clicks / today's impressions — this is what we're
trying to predict; it is NOT a feature.

In [9]:
print("Rows with missing lag features (first day per content):",
      features_df['impressions_lag1'].isna().sum())
print("Total rows:", len(features_df))

Rows with missing lag features (first day per content): 176738
Total rows: 3611061


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
# Check: is each row unique per (client, content, date)?
result1 = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS unique_combinations
    FROM march_data
""")
result1.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┐
│ total_rows │ unique_combinations │
│   int64    │        int64        │
├────────────┼─────────────────────┤
│    9841378 │             9841378 │
└────────────┴─────────────────────┘



In [5]:
result2 = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM march_data
""")
result2.show()

┌───────────┬───────────────┬─────────────┐
│ row_count │ earliest_date │ latest_date │
│   int64   │     date      │    date     │
├───────────┼───────────────┼─────────────┤
│   9841378 │ 2026-03-01    │ 2026-03-31  │
└───────────┴───────────────┴─────────────┘



In [6]:
result3 = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
    FROM march_data
""")
result3.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │
│   int64    │       int64        │
├────────────┼────────────────────┤
│    9841378 │            3611061 │
└────────────┴────────────────────┘



In [10]:
import numpy as np

leak_df = features_df.dropna(subset=['impressions_lag1', 'today_ctr']).copy()

# THE TRAP: this feature is built directly from the label
leak_df['leaky_feature'] = leak_df['today_ctr'] * 1000

print(leak_df[['today_ctr', 'leaky_feature']].head())
print("Rows for experiment:", len(leak_df))

   today_ctr  leaky_feature
1   0.000000       0.000000
2   0.000000       0.000000
3   0.030303      30.303030
4   0.027027      27.027027
5   0.000000       0.000000
Rows for experiment: 3434323


In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Make a simple binary label: is CTR "high" (above median) or not?
median_ctr = leak_df['today_ctr'].median()
leak_df['label_high_ctr'] = (leak_df['today_ctr'] > median_ctr).astype(int)

# ---- ATTEMPT 1: WITH the leaky feature (the trap) ----
X_leaky = leak_df[['impressions_lag1', 'clicks_lag1', 'position_lag1', 'day_of_week', 'leaky_feature']]
y = leak_df['label_high_ctr']

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=42)

model_leaky = LinearRegression()
model_leaky.fit(X_train, y_train)
score_leaky = model_leaky.score(X_test, y_test)

print(f"Score WITH leaky feature (fake, near-perfect): {score_leaky:.4f}")

Score WITH leaky feature (fake, near-perfect): 0.2270


In [12]:
# ---- ATTEMPT 2: WITHOUT the leaky feature (honest) ----
X_honest = leak_df[['impressions_lag1', 'clicks_lag1', 'position_lag1', 'day_of_week']]

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_honest, y, test_size=0.2, random_state=42)

model_honest = LinearRegression()
model_honest.fit(X_train2, y_train2)
score_honest = model_honest.score(X_test2, y_test2)

print(f"Score WITHOUT leaky feature (honest): {score_honest:.4f}")
print(f"\nDifference: leaky score is {score_leaky - score_honest:.4f} points higher — that gap IS the leak.")

Score WITHOUT leaky feature (honest): 0.1477

Difference: leaky score is 0.0794 points higher — that gap IS the leak.


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ---- ATTEMPT 1: WITH the leaky feature ----
X_leaky = leak_df[['impressions_lag1', 'clicks_lag1', 'position_lag1', 'day_of_week', 'leaky_feature']]
y = leak_df['label_high_ctr']

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=42)

clf_leaky = LogisticRegression(max_iter=1000)
clf_leaky.fit(X_train, y_train)
acc_leaky = accuracy_score(y_test, clf_leaky.predict(X_test))

print(f"Accuracy WITH leaky feature (fake, near-perfect): {acc_leaky:.4f}")

Accuracy WITH leaky feature (fake, near-perfect): 0.9999


In [14]:
# ---- ATTEMPT 2: WITHOUT the leaky feature (honest) ----
X_honest = leak_df[['impressions_lag1', 'clicks_lag1', 'position_lag1', 'day_of_week']]

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_honest, y, test_size=0.2, random_state=42)

clf_honest = LogisticRegression(max_iter=1000)
clf_honest.fit(X_train2, y_train2)
acc_honest = accuracy_score(y_test2, clf_honest.predict(X_test2))

print(f"Accuracy WITHOUT leaky feature (honest): {acc_honest:.4f}")
print(f"\nGap: {acc_leaky - acc_honest:.4f} — this jump toward 1.0 IS the leak. The leaky_feature was built directly from today_ctr, so removing it and keeping only yesterday's data gives the real, honest number.")

Accuracy WITHOUT leaky feature (honest): 0.9014

Gap: 0.0986 — this jump toward 1.0 IS the leak. The leaky_feature was built directly from today_ctr, so removing it and keeping only yesterday's data gives the real, honest number.


**The Trap (deliberate leakage demonstration):**

I added `leaky_feature` = today_ctr × 1000 — a column built directly from the
label itself (today's actual CTR), not from anything known before today's
outcome happened.

- Accuracy WITH the leaky feature: 0.9999 (near-perfect — a red flag, not a win)
- Accuracy WITHOUT the leaky feature (honest, using only yesterday's lag features): 0.9014

The jump toward 1.0 is the leakage signature: a model can't genuinely predict
tomorrow with 99.99% accuracy from real-world search signals — that score means
the "future" (today's own outcome) snuck into the inputs. The leaky_feature was
removed, and the honest 0.9014 is the number that reflects what the model can
actually learn from information available at decision time.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation of this slice:**

This data can never tell us WHY a page's CTR changed — only THAT it changed.
There's no content-quality signal, no competitor context, and no information
about Google algorithm updates. Additionally, this slice only used rows where
gsc_data_available IS TRUE (~36.7% of all rows) — the remaining rows likely
represent low-traffic days or clients without active GSC integration that day,
so this analysis may not generalize evenly across all clients. Results here
are observed and directional, not causal or predictive of individual page outcomes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [15]:
# How many unique clients does this slice actually cover?
con.sql("""
    SELECT COUNT(DISTINCT client_hash_id) AS unique_clients
    FROM march_data
    WHERE gsc_data_available IS TRUE
""").show()

┌────────────────┐
│ unique_clients │
│     int64      │
├────────────────┤
│             47 │
└────────────────┘



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.